# Extract an embedding inside OP-TEE, one camera frame at a time

Replay recorded DroneFace probe images. Linux reads one frame, detects a face and aligns a 112 × 112 crop. The trusted application receives that crop and runs SFace on the secure CPU. Its 128 floating-point features exist in secure RAM. After the extraction measurement ends, the trusted application scales and masks the vector, then returns masked coordinates and timing counters to Linux.

The camera link here is simulated with files. Linux can see the source images. An actual camera accessible exclusively to OP-TEE would require a protected capture path.

Use the Jetson directory where the hardware probe succeeded. Install the frame update and select the **Obscura TEE datasets** kernel as described in [the run guide](DATASET_WORKFLOW.md#one-camera-frame-at-a-time).

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import sys

bench = Path.cwd()
if not (bench / "dataset_tee.py").is_file():
    bench = bench / "jetson_benchmark"
sys.path.insert(0, str(bench.resolve()))
from dataset_tee import collect_images, default_ppface, run_dataset

PPFACE = default_ppface()  # Set this to your ppface folder if it is elsewhere on the Jetson.
DATASET = "droneface"  # "lfw" also works for replay.
FRAMES = 30
options = dict(ppface=PPFACE, dataset=DATASET, probes_only=(DATASET == "droneface"),
               profile_extraction=True, power=True, interval_ms=100)
images, inventory = collect_images(PPFACE, DATASET, probes_only=options["probes_only"])
print("Available frames:", len(images), "| First frame:", images[0].key)

def run_frames(name, count, warmup):
    if not 1 <= count <= len(images):
        raise ValueError(f"Choose between 1 and {len(images)} frames")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    folder = bench / "results" / (name + "_" + stamp)
    summary = run_dataset(output=folder, limit=count, warmup=warmup, **options)
    return folder, summary

## 1. Follow one image

This run opens a real TA session and processes one recorded frame, with zero warmups. The session opens before measurement. The row identifies the image, the secure neural computation time, the full extraction interval, and the module energy during that interval. Results are written under `results/`.

In [ ]:
single_folder, single = run_frames("one_frame", 1, warmup=0)
with (single_folder / "frames.csv").open() as stream:
    frame = next(csv.DictReader(stream))
print(json.dumps({key: frame[key] for key in (
    "key", "ta_inference_ms", "extraction_ms", "module_extraction_j",
    "power_samples_in_frame", "energy_status", "secure_workspace_bytes"
)}, indent=2))
print("Results:", single_folder)

## 2. Replay multiple frames

The following run starts a new session, performs two warmups, and then processes the first `FRAMES` distinct files in deterministic filename order. It uses the same session and secure buffers across those measured frames. The one-frame check above is a separate run. Set `FRAMES = len(images)` to include the entire selected stream.

In [ ]:
batch_folder, batch = run_frames("frame_batch", FRAMES, warmup=2)
report = batch["frame_extraction"]
print("Frames measured:", batch["images"])
print("Secure SFace layers (ms):", batch["timings_ms"]["ta_inference_ms"])
print("Crop submitted to embedding ready (ms):", batch["timings_ms"]["extraction_ms"])
print("Module energy during extraction (J/frame):", report["module_extraction_j"])
print("Frames with valid energy:", report["frames_with_energy"], "/", report["frames"])
print("Secure memory:", json.dumps(report["secure_memory"], indent=2))
print("Per-image rows:", batch_folder / "frames.csv")

## Reading the measurements

* `ta_inference_ms` sums the millisecond timers around the SFace layers inside the TA.
* `extraction_ms` runs from submitting the prepared crop through the return of the final SFace layer. This includes secure input preparation, weight transfer and verification, inference and transitions. Face detection precedes this interval. Scaling, masking and file output follow it.
* `module_extraction_j` integrates Jetson module input power over that same interval. The rail includes idle power and other system activity. Record the sample count with the mean and median. A blank entry has its reason in `energy_status`.
* The TA reuses 6,946,808 bytes of workspace for all frames. This is the allocation size for the two activation buffers and the weight buffer. The 512-byte raw embedding occupies part of an activation buffer. The TA writes zero bytes per frame to persistent secure storage. Total peak secure RAM, including the OS and allocator, would need additional instrumentation.

For a repeatable experiment, record the Jetson power mode, cooling and clock settings with the run. Individual rows and the underlying power samples are in `frames.csv` and `power.csv`.